# Ariane — Full Comparison Matrix

**2 LLM methods x 2 baselines x {text, image}**, plus greedy + random-search controls.

| | weak baseline (area, ~1.25e5) | strong baseline (topology, ~8.4e4) |
|---|---|---|
| **LLM region** (assign macros to 3x3 regions) | text / image | text / image |
| **LLM ordering** (propose placement order) | text / image | text / image |

Each LLM cell reports improvement vs ITS OWN baseline (iter-0 greedy in that order). Random-order search is the no-LLM control: if the LLM only matches it, it adds no reasoning.

## Cell 1 - Setup (clone repo, gym shim, protobuf, anthropic)
Needs `strong_search.py` and `matrix_eval.py` pushed to the repo.

In [ ]:
import subprocess, sys, os, shutil, types, glob
import numpy as np

REPO_URL  = "https://github.com/dennis5727/arianePlacement.git"
CLONE_DIR = "/kaggle/working/arianePlacement"
WORK_DIR  = os.path.join(CLONE_DIR, "maskplace")

if not os.path.exists(CLONE_DIR):
    try:
        subprocess.check_call(["git","clone","--depth","1",REPO_URL,CLONE_DIR]); print("cloned")
    except Exception as e: print("git clone failed:", e)
else:
    subprocess.call(["git","-C",CLONE_DIR,"pull","--ff-only"]); print("pulled latest")

if not os.path.exists(WORK_DIR):
    hits=glob.glob("/kaggle/input/**/place_db.py",recursive=True)
    assert hits,"no code via git or dataset"
    shutil.copytree(os.path.dirname(hits[0]),WORK_DIR); print("dataset fallback")

os.chdir(WORK_DIR); sys.path.insert(0,WORK_DIR); print("cwd:",os.getcwd())

os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION","python")
try: subprocess.check_call([sys.executable,"-m","pip","install","-q","protobuf==3.20.3"])
except subprocess.CalledProcessError as e: print("protobuf pin failed:",e)

try:
    import gym; print("real gym:",gym.__version__)
except Exception:
    gym=types.ModuleType("gym"); spaces=types.ModuleType("gym.spaces")
    class _E: pass
    class _D:
        def __init__(s,n): s.n=int(n)
        def contains(s,x):
            try: x=int(x)
            except: return False
            return 0<=x<s.n
    class _B:
        def __init__(s,low=0,high=1,shape=None,dtype=None): s.low,s.high,s.shape,s.dtype=low,high,shape,dtype
    gym.Env=_E; spaces.Discrete=_D; spaces.Box=_B; gym.spaces=spaces
    sys.modules["gym"]=gym; sys.modules["gym.spaces"]=spaces; print("gym shim")

try: subprocess.check_call([sys.executable,"-m","pip","install","-q","anthropic"])
except subprocess.CalledProcessError as e: print("anthropic skipped:",e)

need=["place_db.py","place_env/place_env.py","comp_res.py","greedy_place.py",
      "region_constraint.py","parse_netlist.py","visualize.py","history_tracker.py",
      "llm_interface.py","llm_guided_placement.py","strong_search.py","matrix_eval.py",
      "ariane/netlist.pb.txt"]
miss=[f for f in need if not os.path.exists(f)]
assert not miss, f"MISSING (push to repo): {miss}"
print("files OK\n=== SETUP COMPLETE ===")

## Cell 2 - Sanity check

In [ ]:
from place_db import PlaceDB
placedb=PlaceDB("ariane")
hard=sum(1 for n in placedb.node_info if placedb.node_info[n].get("is_hard"))
print("Nodes",len(placedb.node_info),"| Nets",len(placedb.net_info),
      "| Canvas",placedb.max_height,"| Hard",hard)
assert placedb.max_height==357

## API key (needed for the paid matrix below)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["ANTHROPIC_API_KEY"]=UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
print("API key loaded")

## Cell 3 - Run the full matrix (PAID)
~8 LLM runs (region/order x weak/strong x text/image). Set MODEL/MAX_ITERS to control cost.

In [ ]:
import importlib, matrix_eval, strong_search, llm_guided_placement
importlib.reload(strong_search); importlib.reload(llm_guided_placement); importlib.reload(matrix_eval)
from matrix_eval import evaluate_matrix, print_matrix, save_csv

MODEL     = "claude-sonnet-4-6"   # bump to "claude-opus-4-8" for a stronger (pricier) run
MAX_ITERS = 8                     # LLM iterations per cell
N_RANDOM  = 12                    # no-LLM random-search budget per baseline

# Full matrix: {LLM region, LLM ordering} x {weak area, strong topology} x {text, image}
# plus greedy + random-search controls per baseline. ~8 LLM runs total.
rows = evaluate_matrix(benchmark="ariane", grid=224, model=MODEL,
                       max_iters=MAX_ITERS, patience=3, advisor="claude",
                       n_random=N_RANDOM, outdir="/kaggle/working", verbose=True)

## Results - table, CSV, best layouts

In [ ]:
from matrix_eval import print_matrix, save_csv
print_matrix(rows)
save_csv(rows, "/kaggle/working/matrix.csv")

# show the best layout per (method, baseline) if the figs were saved
from IPython.display import Image, display
import os, glob
for p in sorted(glob.glob("/kaggle/working/*_region_*.png")+glob.glob("/kaggle/working/*_order_*.png")):
    print(os.path.basename(p)); display(Image(p))